# Part 5: The Transformer Block

## Putting It All Together

A Transformer block combines several components:
1. **Multi-Head Attention** (from previous notebook)
2. **Feed-Forward Network** (simple MLP)
3. **Layer Normalization** (stabilizes training)
4. **Residual Connections** (helps gradient flow)

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


## Component 1: Layer Normalization

### Why Normalize?

Neural networks train better when inputs have consistent statistics. Layer normalization ensures each layer's inputs have mean=0 and std=1.

### Layer Norm vs Batch Norm

- **Batch Norm**: Normalize across the batch dimension
- **Layer Norm**: Normalize across the feature dimension

Layer Norm is preferred in Transformers because:
1. Works with variable sequence lengths
2. No dependency on batch statistics
3. Better for sequential models


In [ ]:
class LayerNorm:
    """
    Layer Normalization: Normalize across features (last dimension).
    
    For input of shape (seq_len, d_model):
    - Compute mean and variance across d_model for each position
    - Normalize to mean=0, var=1
    - Scale and shift with learnable parameters
    """
    
    def __init__(self, d_model, eps=1e-6):
        self.eps = eps
        self.gamma = np.ones(d_model)   # Learnable scale
        self.beta = np.zeros(d_model)   # Learnable shift
    
    def forward(self, x):
        """
        x: (seq_len, d_model) or (batch, seq_len, d_model)
        """
        # Compute mean and variance along last dimension
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        
        # Normalize
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        
        # Scale and shift
        return self.gamma * x_norm + self.beta

# Demonstrate
d_model = 8
seq_len = 4

# Random input with different statistics at each position
x = np.random.randn(seq_len, d_model) * 3 + 5  # Mean ~5, std ~3

ln = LayerNorm(d_model)
x_normed = ln.forward(x)

print("Before LayerNorm:")
print(f"  Mean per position: {x.mean(axis=-1).round(2)}")
print(f"  Std per position:  {x.std(axis=-1).round(2)}")

print("\nAfter LayerNorm:")
print(f"  Mean per position: {x_normed.mean(axis=-1).round(2)}")
print(f"  Std per position:  {x_normed.std(axis=-1).round(2)}")


In [ ]:
# Visualize the effect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
im = ax.imshow(x, cmap='RdBu', aspect='auto')
ax.set_title('Before LayerNorm\n(Varied statistics)')
ax.set_xlabel('Feature Dimension')
ax.set_ylabel('Position')
plt.colorbar(im, ax=ax)

ax = axes[1]
im = ax.imshow(x_normed, cmap='RdBu', aspect='auto')
ax.set_title('After LayerNorm\n(Normalized per position)')
ax.set_xlabel('Feature Dimension')
ax.set_ylabel('Position')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()


## Component 2: Feed-Forward Network (FFN)

### The "Position-wise" MLP

After attention, each position goes through a simple 2-layer MLP independently:

```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
```

### Key Details

- **Inner dimension**: Usually 4x the model dimension (e.g., 512 -> 2048 -> 512)
- **Position-wise**: Same MLP applied to each position independently
- **Activation**: ReLU in original paper, GELU in modern models


In [ ]:
def relu(x):
    return np.maximum(0, x)

def gelu(x):
    """Gaussian Error Linear Unit - smoother than ReLU"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


class FeedForward:
    """
    Position-wise Feed-Forward Network.
    
    Two linear transformations with activation in between.
    """
    
    def __init__(self, d_model, d_ff=None, activation='relu'):
        """
        d_model: Model dimension
        d_ff: Inner dimension (default: 4 * d_model)
        """
        if d_ff is None:
            d_ff = 4 * d_model
        
        self.d_model = d_model
        self.d_ff = d_ff
        
        # First layer expands dimension
        self.W1 = np.random.randn(d_model, d_ff) * 0.02
        self.b1 = np.zeros(d_ff)
        
        # Second layer projects back
        self.W2 = np.random.randn(d_ff, d_model) * 0.02
        self.b2 = np.zeros(d_model)
        
        self.activation = gelu if activation == 'gelu' else relu
    
    def forward(self, x):
        """
        x: (seq_len, d_model)
        returns: (seq_len, d_model)
        """
        # Expand: d_model -> d_ff
        hidden = self.activation(x @ self.W1 + self.b1)
        
        # Project back: d_ff -> d_model
        output = hidden @ self.W2 + self.b2
        
        return output

# Demonstrate
d_model = 8
d_ff = 32  # 4x expansion
seq_len = 4

ffn = FeedForward(d_model, d_ff)
x = np.random.randn(seq_len, d_model)

output = ffn.forward(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nInner dimension: {d_ff} (4x expansion)")
print(f"Same input/output dimension - this is a 'residual-friendly' design")


In [ ]:
# Visualize ReLU vs GELU
x_range = np.linspace(-3, 3, 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_range, relu(x_range), label='ReLU', lw=2)
ax.plot(x_range, gelu(x_range), label='GELU', lw=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Input')
ax.set_ylabel('Output')
ax.set_title('Activation Functions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("GELU is smoother than ReLU and often works better for Transformers")
print("GPT and BERT use GELU; original Transformer used ReLU")


## Component 3: Residual Connections

### The Problem

Deep networks suffer from vanishing gradients - the error signal weakens as it flows backward.

### The Solution

Add the input directly to the output:

```
output = sublayer(x) + x
```

This creates a "gradient highway" - even if the sublayer's gradients are small, the gradient can flow through the skip connection.


In [ ]:
# Visualize residual connections
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without residual
ax = axes[0]
layers = ['Input', 'Layer 1', 'Layer 2', 'Layer 3', 'Output']
y_pos = [0, 1, 2, 3, 4]

for i in range(len(layers) - 1):
    ax.annotate('', xy=(0.5, y_pos[i+1]-0.1), xytext=(0.5, y_pos[i]+0.1),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))

for i, (layer, y) in enumerate(zip(layers, y_pos)):
    ax.add_patch(plt.Rectangle((0.2, y-0.08), 0.6, 0.16, 
                                facecolor='lightblue', edgecolor='black'))
    ax.text(0.5, y, layer, ha='center', va='center', fontsize=10)

ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 4.5)
ax.axis('off')
ax.set_title('Without Residual Connections\n(Gradients must flow through all layers)')

# With residual
ax = axes[1]
for i in range(len(layers) - 1):
    # Direct path
    ax.annotate('', xy=(0.5, y_pos[i+1]-0.1), xytext=(0.5, y_pos[i]+0.1),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    # Skip connection
    if i < len(layers) - 2:  # Not for input
        ax.annotate('', xy=(0.82, y_pos[i+1]-0.05), xytext=(0.82, y_pos[i]+0.05),
                    arrowprops=dict(arrowstyle='->', color='green', lw=2, 
                                    connectionstyle='arc3,rad=-0.3'))

for i, (layer, y) in enumerate(zip(layers, y_pos)):
    ax.add_patch(plt.Rectangle((0.2, y-0.08), 0.6, 0.16, 
                                facecolor='lightblue', edgecolor='black'))
    ax.text(0.5, y, layer, ha='center', va='center', fontsize=10)

ax.set_xlim(0, 1.2)
ax.set_ylim(-0.5, 4.5)
ax.axis('off')
ax.set_title('With Residual Connections\n(Gradients can skip layers via green paths)')

plt.tight_layout()
plt.show()

print("Key insight: Gradients always have a direct path to earlier layers")


## The Complete Transformer Block

Now let's assemble everything:

```
Input
  |
  v
[Multi-Head Attention]
  |
  +---> Add (residual)
  |
  v
[Layer Norm]
  |
  v
[Feed-Forward Network]
  |
  +---> Add (residual)
  |
  v
[Layer Norm]
  |
  v
Output
```

This pattern is called **Post-LN** (normalization after residual). There's also **Pre-LN** which normalizes before the sublayer.


In [ ]:
# Multi-Head Attention (from previous notebook)
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = np.random.randn(d_model, d_model) * 0.02
        self.W_K = np.random.randn(d_model, d_model) * 0.02
        self.W_V = np.random.randn(d_model, d_model) * 0.02
        self.W_O = np.random.randn(d_model, d_model) * 0.02
    
    def forward(self, X, mask=None):
        seq_len = X.shape[0]
        
        Q = X @ self.W_Q
        K = X @ self.W_K
        V = X @ self.W_V
        
        Q = Q.reshape(seq_len, self.num_heads, self.d_k).transpose(1, 0, 2)
        K = K.reshape(seq_len, self.num_heads, self.d_k).transpose(1, 0, 2)
        V = V.reshape(seq_len, self.num_heads, self.d_k).transpose(1, 0, 2)
        
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)
        
        attention_weights = softmax(scores, axis=-1)
        context = np.matmul(attention_weights, V)
        
        context = context.transpose(1, 0, 2).reshape(seq_len, self.d_model)
        output = context @ self.W_O
        
        return output, attention_weights


class TransformerBlock:
    """
    A complete Transformer block.
    
    Components:
    1. Multi-Head Self-Attention
    2. Add & Norm
    3. Feed-Forward Network
    4. Add & Norm
    """
    
    def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = dropout  # Would be used during training
    
    def forward(self, x, mask=None):
        """
        x: (seq_len, d_model)
        mask: optional attention mask
        
        Returns: (seq_len, d_model)
        """
        # Multi-Head Attention + Residual + LayerNorm
        attn_output, attn_weights = self.attention.forward(x, mask)
        x = self.norm1.forward(x + attn_output)  # Residual + Norm
        
        # Feed-Forward + Residual + LayerNorm
        ffn_output = self.ffn.forward(x)
        x = self.norm2.forward(x + ffn_output)  # Residual + Norm
        
        return x, attn_weights

# Test the complete block
d_model = 64
num_heads = 4
seq_len = 10

block = TransformerBlock(d_model, num_heads)
x = np.random.randn(seq_len, d_model)

output, attn_weights = block.forward(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nThe block preserves the input dimensions!")
print(f"This allows stacking multiple blocks.")


In [ ]:
class PreLNTransformerBlock:
    """
    Transformer block with Pre-Layer Normalization.
    Often used in GPT-style models.
    """
    
    def __init__(self, d_model, num_heads, d_ff=None):
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Pre-LN: Normalize BEFORE sublayer
        attn_output, attn_weights = self.attention.forward(self.norm1.forward(x), mask)
        x = x + attn_output  # Residual
        
        ffn_output = self.ffn.forward(self.norm2.forward(x))
        x = x + ffn_output  # Residual
        
        return x, attn_weights

# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Post-LN diagram
ax = axes[0]
components = ['Input', 'Attention', 'Add', 'Norm', 'FFN', 'Add', 'Norm', 'Output']
y_pos = list(range(len(components)))

for i, (comp, y) in enumerate(zip(components, y_pos)):
    color = 'lightblue' if comp in ['Input', 'Output'] else \
            'lightgreen' if comp in ['Attention', 'FFN'] else \
            'lightyellow' if comp == 'Norm' else 'lightcoral'
    ax.add_patch(plt.Rectangle((0.3, y-0.15), 0.4, 0.3, 
                                facecolor=color, edgecolor='black'))
    ax.text(0.5, y, comp, ha='center', va='center', fontsize=10)

ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 7.5)
ax.axis('off')
ax.set_title('Post-LN\n(Original Transformer)')

# Pre-LN diagram
ax = axes[1]
components = ['Input', 'Norm', 'Attention', 'Add', 'Norm', 'FFN', 'Add', 'Output']

for i, (comp, y) in enumerate(zip(components, y_pos)):
    color = 'lightblue' if comp in ['Input', 'Output'] else \
            'lightgreen' if comp in ['Attention', 'FFN'] else \
            'lightyellow' if comp == 'Norm' else 'lightcoral'
    ax.add_patch(plt.Rectangle((0.3, y-0.15), 0.4, 0.3, 
                                facecolor=color, edgecolor='black'))
    ax.text(0.5, y, comp, ha='center', va='center', fontsize=10)

ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 7.5)
ax.axis('off')
ax.set_title('Pre-LN\n(GPT-style)')

plt.tight_layout()
plt.show()


In [ ]:
class TransformerEncoder:
    """
    Stack of Transformer blocks.
    """
    
    def __init__(self, d_model, num_heads, num_layers, d_ff=None):
        self.layers = [
            TransformerBlock(d_model, num_heads, d_ff) 
            for _ in range(num_layers)
        ]
    
    def forward(self, x, mask=None):
        """
        Pass input through all transformer blocks.
        """
        all_attentions = []
        
        for layer in self.layers:
            x, attn = layer.forward(x, mask)
            all_attentions.append(attn)
        
        return x, all_attentions

# Create a 4-layer transformer
d_model = 64
num_heads = 4
num_layers = 4
seq_len = 10

encoder = TransformerEncoder(d_model, num_heads, num_layers)
x = np.random.randn(seq_len, d_model)

output, attentions = encoder.forward(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Number of attention maps: {len(attentions)}")
print(f"\nThe representation is refined through {num_layers} layers")


In [ ]:
# Visualize attention patterns at different layers
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, attn in enumerate(attentions):
    ax = axes[i]
    # Average across heads
    avg_attn = attn.mean(axis=0)
    im = ax.imshow(avg_attn, cmap='Blues')
    ax.set_title(f'Layer {i+1}\n(avg across heads)')
    ax.set_xlabel('To')
    if i == 0:
        ax.set_ylabel('From')

plt.suptitle('Attention Patterns at Different Layers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Deeper layers often develop more specialized attention patterns")


## Summary: Transformer Block

### Components

| Component | Purpose |
|-----------|---------|
| Multi-Head Attention | Mix information between positions |
| Feed-Forward Network | Process each position independently |
| Layer Normalization | Stabilize activations |
| Residual Connections | Enable gradient flow |

### The Formula

For Post-LN:
```
x = LayerNorm(x + MultiHeadAttention(x))
x = LayerNorm(x + FeedForward(x))
```

For Pre-LN:
```
x = x + MultiHeadAttention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))
```

### Key Properties

1. **Preserves dimensions**: Input and output have same shape
2. **Stackable**: Can build arbitrarily deep models
3. **Position-wise processing**: FFN processes each position independently
4. **Global mixing**: Attention connects all positions

---

**Next: 06_full_transformer.ipynb** - The complete Transformer architecture!
